In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/01-config

In [0]:
# COMMAND ----------

class Upserter:

    def __init__(self, merge_query, temp_view_name):
        self.merge_query = merge_query
        self.temp_view_name = temp_view_name

    def upsert(self, df_micro_batch, batch_id):

        df_micro_batch.createOrReplaceTempView(
            self.temp_view_name
        )

        spark.sql(self.merge_query)

### Description de la classe `Upserter`

La classe `Upserter` sert à appliquer un **MERGE SQL** sur chaque micro-batch reçu par un flux Spark Structured Streaming.

Son fonctionnement est simple :

`Streaming DataFrame → foreachBatch() → Temp View → MERGE SQL → Table Silver`

Le constructeur reçoit deux éléments :

- `merge_query` : la requête SQL `MERGE` à exécuter.
- `temp_view_name` : le nom de la vue temporaire utilisée comme source du `MERGE`.

La méthode `upsert()` reçoit ensuite chaque micro-batch du stream dans `df_micro_batch`.

Elle réalise deux actions :

1. `createOrReplaceTempView()` transforme le micro-batch en vue temporaire SQL.
2. `spark.sql(self.merge_query)` exécute la requête `MERGE` entre cette vue temporaire et la table Silver cible.

Cette classe permet donc de réutiliser la même logique d’**upsert** pour plusieurs tables Silver sans réécrire le mécanisme de traitement à chaque fois.

Un **upsert** signifie :

`UPDATE si l’enregistrement existe déjà + INSERT s’il n’existe pas`

Cette approche est particulièrement utile pour transformer progressivement les nouvelles données de la couche Bronze vers la couche Silver.

In [0]:
# COMMAND ----------

class CDCUpserter:

    def __init__(
        self,
        merge_query,
        temp_view_name,
        id_column,
        sort_by
    ):
        self.merge_query = merge_query
        self.temp_view_name = temp_view_name
        self.id_column = id_column
        self.sort_by = sort_by

    def upsert(self, df_micro_batch, batch_id):

        from pyspark.sql.window import Window
        from pyspark.sql import functions as F

        window = (
            Window
            .partitionBy(self.id_column)
            .orderBy(F.col(self.sort_by).desc())
        )

        (
            df_micro_batch
            .filter(
                F.col("update_type").isin(
                    ["new", "update"]
                )
            )
            .withColumn(
                "rank",
                F.row_number().over(window)
            )
            .filter("rank = 1")
            .drop("rank")
            .createOrReplaceTempView(
                self.temp_view_name
            )
        )

        spark.sql(self.merge_query)

### Description de la classe `CDCUpserter`

La classe `CDCUpserter` sert à gérer les données **CDC (Change Data Capture)** provenant du profil utilisateur.

Son objectif est de conserver, pour chaque utilisateur, **la version la plus récente de ses informations** avant d’exécuter le `MERGE` vers la table Silver.

Le fonctionnement est :

`CDC Micro-batch → Filtrer new/update → Garder la dernière version → Temp View → MERGE → Table Silver`

La classe reçoit plusieurs paramètres :

- `merge_query` : requête SQL `MERGE` utilisée pour mettre à jour la table cible.
- `temp_view_name` : nom de la vue temporaire créée à partir du micro-batch.
- `id_column` : colonne utilisée pour identifier l’entité, par exemple `user_id`.
- `sort_by` : colonne utilisée pour déterminer la version la plus récente, par exemple `updated`.

La méthode `upsert()` effectue plusieurs traitements :

1. elle conserve uniquement les événements `new` et `update` ;
2. elle regroupe les données par utilisateur grâce à une fenêtre Spark ;
3. elle trie les modifications de la plus récente à la plus ancienne ;
4. elle garde uniquement la dernière version de chaque utilisateur ;
5. elle crée une vue temporaire ;
6. elle exécute ensuite le `MERGE` SQL vers la table Silver.

Cette logique permet d’éviter d’appliquer plusieurs anciennes versions du même profil lorsqu’elles arrivent dans un même micro-batch.

Le principe est donc :

`Plusieurs modifications d’un utilisateur → dernière modification conservée → mise à jour de la table Silver`

Cette classe est principalement utilisée dans le projet pour traiter les événements CDC du `user_profile`.

In [0]:
# COMMAND ----------

class Silver:

    def __init__(self, catalog="dev"):

        self.conf = Config()

        self.checkpoint_base = (
            self.conf.base_dir_checkpoint
            + "/checkpoints"
        )

        self.catalog = catalog

        self.bronze = self.conf.bronze_schema
        self.silver = self.conf.silver_schema

        self.maxFilesPerTrigger = (
            self.conf.maxFilesPerTrigger
        )
    def _await_queries(self, once):
        if once:
            for stream in spark.streams.active:
                stream.awaitTermination()
    def upsert(
    self,
    once=True,
    processing_time="5 seconds"):

        import time

        start = int(time.time())

        print("\nExecuting silver layer upsert...")

        self.upsert_users(once, processing_time)
        self.upsert_gym_logs(once, processing_time)
        self.upsert_user_profile(once, processing_time)
        self.upsert_workouts(once, processing_time)
        self.upsert_heart_rate(once, processing_time)

        self._await_queries(once)

        self.upsert_user_bins(once, processing_time)
        self.upsert_completed_workouts(once, processing_time)

        self._await_queries(once)

        self.upsert_workout_bpm(once, processing_time)

        self._await_queries(once)

        print(
            f"Silver completed in "
            f"{int(time.time()) - start} seconds"
        )
    #############################
    def upsert_users(
        self,
        once=True,
        processing_time="15 seconds",
        startingVersion=0
    ):

        query = f"""
            MERGE INTO
                {self.catalog}.{self.silver}.users a

            USING users_delta b

            ON a.user_id = b.user_id

            WHEN NOT MATCHED
            THEN INSERT *
        """

        data_upserter = Upserter(
            query,
            "users_delta"
        )

        df_delta = (
            spark.readStream
            .option(
                "startingVersion",
                startingVersion
            )
            .table(
                f"{self.catalog}."
                f"{self.bronze}."
                f"registered_users_bz"
            )
            .selectExpr(
                "user_id",
                "device_id",
                "mac_address",
                """
                cast(
                    registration_timestamp
                    as timestamp
                ) as registration_timestamp
                """
            )
            .withWatermark(
                "registration_timestamp",
                "30 seconds"
            )
            .dropDuplicates(
                ["user_id", "device_id"]
            )
        )

        stream_writer = (
            df_delta.writeStream
            .foreachBatch(
                data_upserter.upsert
            )
            .outputMode("update")
            .option(
                "checkpointLocation",
                f"{self.checkpoint_base}/users"
            )
            .queryName(
                "users_upsert_stream"
            )
        )

        if once:
            return (
                stream_writer
                .trigger(availableNow=True)
                .start()
            )

        return (
            stream_writer
            .trigger(
                processingTime=processing_time
            )
            .start()
        )
    ########################################################################
    def upsert_gym_logs(
        self,
        once=True,
        processing_time="15 seconds",
        startingVersion=0
    ):

        query = f"""
            MERGE INTO
                {self.catalog}.{self.silver}.gym_logs a

            USING gym_logs_delta b

            ON  a.mac_address = b.mac_address
            AND a.gym = b.gym
            AND a.login = b.login

            WHEN MATCHED
                 AND b.logout > a.login
                 AND (
                     a.logout IS NULL
                     OR b.logout > a.logout
                 )
            THEN UPDATE SET
                 logout = b.logout

            WHEN NOT MATCHED
            THEN INSERT *
        """

        data_upserter = Upserter(
            query,
            "gym_logs_delta"
        )

        df_delta = (
            spark.readStream
            .option(
                "startingVersion",
                startingVersion
            )
            .table(
                f"{self.catalog}."
                f"{self.bronze}."
                f"gym_logins_bz"
            )
            .selectExpr(
                "mac_address",
                "gym",
                "cast(login as timestamp) as login",
                "cast(logout as timestamp) as logout"
            )
            .withWatermark(
                "login",
                "30 seconds"
            )
            .dropDuplicates(
                ["mac_address", "gym", "login"]
            )
        )

        stream_writer = (
            df_delta.writeStream
            .foreachBatch(
                data_upserter.upsert
            )
            .outputMode("update")
            .option(
                "checkpointLocation",
                f"{self.checkpoint_base}/gym_logs"
            )
            .queryName(
                "gym_logs_upsert_stream"
            )
        )

        if once:
            return (
                stream_writer
                .trigger(availableNow=True)
                .start()
            )

        return (
            stream_writer
            .trigger(
                processingTime=processing_time
            )
            .start()
        )
    ##################################
    def upsert_user_profile(
        self,
        once=True,
        processing_time="15 seconds",
        startingVersion=0
    ):

        from pyspark.sql import functions as F

        schema = """
            user_id BIGINT,
            update_type STRING,
            timestamp FLOAT,
            dob STRING,
            sex STRING,
            gender STRING,
            first_name STRING,
            last_name STRING,
            address STRUCT<
                street_address: STRING,
                city: STRING,
                state: STRING,
                zip: INT
            >
        """

        query = f"""
            MERGE INTO
                {self.catalog}.{self.silver}.user_profile a

            USING user_profile_cdc b

            ON a.user_id = b.user_id

            WHEN MATCHED
                 AND a.updated < b.updated
            THEN UPDATE SET *

            WHEN NOT MATCHED
            THEN INSERT *
        """

        data_upserter = CDCUpserter(
            query,
            "user_profile_cdc",
            "user_id",
            "updated"
        )

        df_cdc = (
            spark.readStream
            .option(
                "startingVersion",
                startingVersion
            )
            .table(
                f"{self.catalog}."
                f"{self.bronze}."
                f"kafka_multiplex_bz"
            )
            .filter(
                "topic = 'user_info'"
            )
            .select(
                F.from_json(
                    F.col("value"),
                    schema
                ).alias("v")
            )
            .select("v.*")
            .select(
                "user_id",

                F.to_date(
                    "dob",
                    "MM/dd/yyyy"
                ).alias("dob"),

                "sex",
                "gender",
                "first_name",
                "last_name",

                F.col(
                    "address.street_address"
                ).alias("street_address"),

                F.col(
                    "address.city"
                ).alias("city"),

                F.col(
                    "address.state"
                ).alias("state"),

                F.col(
                    "address.zip"
                ).alias("zip"),

                F.col("timestamp")
                .cast("timestamp")
                .alias("updated"),

                "update_type"
            )
            .withWatermark(
                "updated",
                "30 seconds"
            )
            .dropDuplicates(
                ["user_id", "updated"]
            )
        )

        stream_writer = (
            df_cdc.writeStream
            .foreachBatch(
                data_upserter.upsert
            )
            .outputMode("update")
            .option(
                "checkpointLocation",
                f"{self.checkpoint_base}/user_profile"
            )
            .queryName(
                "user_profile_stream"
            )
        )

        if once:
            return (
                stream_writer
                .trigger(availableNow=True)
                .start()
            )

        return (
            stream_writer
            .trigger(
                processingTime=processing_time
            )
            .start()
        )
    #########################################################
    def upsert_workouts(
        self,
        once=True,
        processing_time="10 seconds",
        startingVersion=0
    ):

        from pyspark.sql import functions as F

        schema = """
            user_id INT,
            workout_id INT,
            timestamp FLOAT,
            action STRING,
            session_id INT
        """

        query = f"""
            MERGE INTO
                {self.catalog}.{self.silver}.workouts a

            USING workouts_delta b

            ON  a.user_id = b.user_id
            AND a.time = b.time

            WHEN NOT MATCHED
            THEN INSERT *
        """

        data_upserter = Upserter(
            query,
            "workouts_delta"
        )

        df_delta = (
            spark.readStream
            .option(
                "startingVersion",
                startingVersion
            )
            .table(
                f"{self.catalog}."
                f"{self.bronze}."
                f"kafka_multiplex_bz"
            )
            .filter(
                "topic = 'workout'"
            )
            .select(
                F.from_json(
                    F.col("value"),
                    schema
                ).alias("v")
            )
            .select("v.*")
            .select(
                "user_id",
                "workout_id",

                F.col("timestamp")
                .cast("timestamp")
                .alias("time"),

                "action",
                "session_id"
            )
            .withWatermark(
                "time",
                "30 seconds"
            )
            .dropDuplicates(
                ["user_id", "time"]
            )
        )

        stream_writer = (
            df_delta.writeStream
            .foreachBatch(
                data_upserter.upsert
            )
            .outputMode("update")
            .option(
                "checkpointLocation",
                f"{self.checkpoint_base}/workouts"
            )
            .queryName(
                "workouts_upsert_stream"
            )
        )

        if once:
            return (
                stream_writer
                .trigger(availableNow=True)
                .start()
            )

        return (
            stream_writer
            .trigger(
                processingTime=processing_time
            )
            .start()
        )

    ###############################################################################
    def upsert_heart_rate(
        self,
        once=True,
        processing_time="10 seconds",
        startingVersion=0
    ):

        from pyspark.sql import functions as F

        schema = """
            device_id LONG,
            time TIMESTAMP,
            heartrate DOUBLE
        """

        query = f"""
            MERGE INTO
                {self.catalog}.{self.silver}.heart_rate a

            USING heart_rate_delta b

            ON  a.device_id = b.device_id
            AND a.time = b.time

            WHEN NOT MATCHED
            THEN INSERT *
        """

        data_upserter = Upserter(
            query,
            "heart_rate_delta"
        )

        df_delta = (
            spark.readStream
            .option(
                "startingVersion",
                startingVersion
            )
            .table(
                f"{self.catalog}."
                f"{self.bronze}."
                f"kafka_multiplex_bz"
            )
            .filter(
                "topic = 'bpm'"
            )
            .select(
                F.from_json(
                    F.col("value"),
                    schema
                ).alias("v")
            )
            .select(
                "v.*",

                F.when(
                    F.col("v.heartrate") <= 0,
                    False
                )
                .otherwise(True)
                .alias("valid")
            )
            .withWatermark(
                "time",
                "30 seconds"
            )
            .dropDuplicates(
                ["device_id", "time"]
            )
        )

        stream_writer = (
            df_delta.writeStream
            .foreachBatch(
                data_upserter.upsert
            )
            .outputMode("update")
            .option(
                "checkpointLocation",
                f"{self.checkpoint_base}/heart_rate"
            )
            .queryName(
                "heart_rate_upsert_stream"
            )
        )

        if once:
            return (
                stream_writer
                .trigger(availableNow=True)
                .start()
            )

        return (
            stream_writer
            .trigger(
                processingTime=processing_time
            )
            .start()
        )
    ###############################################


    def age_bins(self, dob_col):

        from pyspark.sql import functions as F

        age = F.floor(
            F.months_between(
                F.current_date(),
                dob_col
            ) / 12
        )

        return (
            F.when(age < 18, "under 18")
            .when(
                (age >= 18) & (age < 25),
                "18-25"
            )
            .when(
                (age >= 25) & (age < 35),
                "25-35"
            )
            .when(
                (age >= 35) & (age < 45),
                "35-45"
            )
            .when(
                (age >= 45) & (age < 55),
                "45-55"
            )
            .when(
                (age >= 55) & (age < 65),
                "55-65"
            )
            .when(
                (age >= 65) & (age < 75),
                "65-75"
            )
            .when(
                (age >= 75) & (age < 85),
                "75-85"
            )
            .when(
                (age >= 85) & (age < 95),
                "85-95"
            )
            .when(age >= 95, "95+")
            .otherwise("invalid age")
            .alias("age")
        )
    ######################################################










    def upsert_user_bins(
        self,
        once=True,
        processing_time="15 seconds",
        startingVersion=0
    ):

        from pyspark.sql import functions as F

        query = f"""
            MERGE INTO
                {self.catalog}.{self.silver}.user_bins a

            USING user_bins_delta b

            ON a.user_id = b.user_id

            WHEN MATCHED
            THEN UPDATE SET *

            WHEN NOT MATCHED
            THEN INSERT *
        """

        data_upserter = Upserter(
            query,
            "user_bins_delta"
        )

        df_users = (
            spark.table(
                f"{self.catalog}."
                f"{self.silver}.users"
            )
            .select("user_id")
        )

        df_delta = (
            spark.readStream
            .option(
                "startingVersion",
                startingVersion
            )
            .option(
                "ignoreChanges",
                True
            )
            .table(
                f"{self.catalog}."
                f"{self.silver}."
                f"user_profile"
            )
            .join(
                df_users,
                ["user_id"],
                "left"
            )
            .select(
                "user_id",
                self.age_bins(
                    F.col("dob")
                ),
                "gender",
                "city",
                "state"
            )
        )

        stream_writer = (
            df_delta.writeStream
            .foreachBatch(
                data_upserter.upsert
            )
            .outputMode("update")
            .option(
                "checkpointLocation",
                f"{self.checkpoint_base}/user_bins"
            )
            .queryName(
                "user_bins_upsert_stream"
            )
        )

        if once:
            return (
                stream_writer
                .trigger(availableNow=True)
                .start()
            )

        return (
            stream_writer
            .trigger(
                processingTime=processing_time
            )
            .start()
        )
    ######################################################################################
    def upsert_completed_workouts(
        self,
        once=True,
        processing_time="15 seconds",
        startingVersion=0
    ):

        from pyspark.sql import functions as F

        query = f"""
            MERGE INTO
                {self.catalog}.{self.silver}.completed_workouts a

            USING completed_workouts_delta b

            ON  a.user_id = b.user_id
            AND a.workout_id = b.workout_id
            AND a.session_id = b.session_id

            WHEN NOT MATCHED
            THEN INSERT *
        """

        data_upserter = Upserter(
            query,
            "completed_workouts_delta"
        )

        source = (
            f"{self.catalog}."
            f"{self.silver}."
            f"workouts"
        )

        df_start = (
            spark.readStream
            .option(
                "startingVersion",
                startingVersion
            )
            .table(source)
            .filter(
                "action = 'start'"
            )
            .selectExpr(
                "user_id",
                "workout_id",
                "session_id",
                "time as start_time"
            )
            .withWatermark(
                "start_time",
                "30 seconds"
            )
        )

        df_stop = (
            spark.readStream
            .option(
                "startingVersion",
                startingVersion
            )
            .table(source)
            .filter(
                "action = 'stop'"
            )
            .selectExpr(
                "user_id",
                "workout_id",
                "session_id",
                "time as end_time"
            )
            .withWatermark(
                "end_time",
                "30 seconds"
            )
        )

        join_condition = [
            df_start.user_id
            == df_stop.user_id,

            df_start.workout_id
            == df_stop.workout_id,

            df_start.session_id
            == df_stop.session_id,

            df_stop.end_time
            >= df_start.start_time,

            df_stop.end_time
            < df_start.start_time
            + F.expr("interval 3 hours")
        ]

        df_delta = (
            df_start
            .join(
                df_stop,
                join_condition
            )
            .select(
                df_start.user_id,
                df_start.workout_id,
                df_start.session_id,
                df_start.start_time,
                df_stop.end_time
            )
        )

        stream_writer = (
            df_delta.writeStream
            .foreachBatch(
                data_upserter.upsert
            )
            .outputMode("append")
            .option(
                "checkpointLocation",
                f"{self.checkpoint_base}/completed_workouts"
            )
            .queryName(
                "completed_workouts_upsert_stream"
            )
        )

        if once:
            return (
                stream_writer
                .trigger(availableNow=True)
                .start()
            )

        return (
            stream_writer
            .trigger(
                processingTime=processing_time
            )
            .start()
        )
    ############################################################################################
    def upsert_workout_bpm(
        self,
        once=True,
        processing_time="15 seconds",
        startingVersion=0
    ):

        from pyspark.sql import functions as F

        query = f"""
            MERGE INTO
                {self.catalog}.{self.silver}.workout_bpm a

            USING workout_bpm_delta b

            ON  a.user_id = b.user_id
            AND a.workout_id = b.workout_id
            AND a.session_id = b.session_id
            AND a.time = b.time

            WHEN NOT MATCHED
            THEN INSERT *
        """

        data_upserter = Upserter(
            query,
            "workout_bpm_delta"
        )

        df_users = (
            spark.table(
                f"{self.catalog}."
                f"{self.silver}.users"
            )
        )

        df_completed = (
            spark.readStream
            .option(
                "startingVersion",
                startingVersion
            )
            .table(
                f"{self.catalog}."
                f"{self.silver}."
                f"completed_workouts"
            )
            .join(
                df_users,
                "user_id"
            )
            .selectExpr(
                "user_id",
                "device_id",
                "workout_id",
                "session_id",
                "start_time",
                "end_time"
            )
            .withWatermark(
                "end_time",
                "30 seconds"
            )
        )

        df_bpm = (
            spark.readStream
            .option(
                "startingVersion",
                startingVersion
            )
            .table(
                f"{self.catalog}."
                f"{self.silver}.heart_rate"
            )
            .filter(
                "valid = true"
            )
            .selectExpr(
                "device_id",
                "time",
                "heartrate"
            )
            .withWatermark(
                "time",
                "30 seconds"
            )
        )

        condition = [

            df_completed.device_id
            == df_bpm.device_id,

            df_bpm.time
            > df_completed.start_time,

            df_bpm.time
            <= df_completed.end_time,

            df_completed.end_time
            < df_bpm.time
            + F.expr("interval 3 hours")
        ]

        df_delta = (
            df_bpm
            .join(
                df_completed,
                condition
            )
            .select(
                "user_id",
                "workout_id",
                "session_id",
                "start_time",
                "end_time",
                "time",
                "heartrate"
            )
        )

        stream_writer = (
            df_delta.writeStream
            .foreachBatch(
                data_upserter.upsert
            )
            .outputMode("append")
            .option(
                "checkpointLocation",
                f"{self.checkpoint_base}/workout_bpm"
            )
            .queryName(
                "workout_bpm_upsert_stream"
            )
        )

        if once:
            return (
                stream_writer
                .trigger(availableNow=True)
                .start()
            )

        return (
            stream_writer
            .trigger(
                processingTime=processing_time
            )
            .start()
        )
    ###################################################################
    def _await_queries(self, once):

        if once:
            for stream in spark.streams.active:
                stream.awaitTermination()


    def upsert(
        self,
        once=True,
        processing_time="5 seconds"
    ):

        import time

        start = int(time.time())

        print(
            "\nExecuting Silver layer..."
        )

        # Première dépendance
        self.upsert_users(
            once,
            processing_time
        )

        self.upsert_gym_logs(
            once,
            processing_time
        )

        self.upsert_user_profile(
            once,
            processing_time
        )

        self.upsert_workouts(
            once,
            processing_time
        )

        self.upsert_heart_rate(
            once,
            processing_time
        )

        self._await_queries(once)

        print(
            "Silver stage 1 completed."
        )

        # Dépendent des tables précédentes
        self.upsert_user_bins(
            once,
            processing_time
        )

        self.upsert_completed_workouts(
            once,
            processing_time
        )

        self._await_queries(once)

        print(
            "Silver stage 2 completed."
        )

        # Dépend de heart_rate +
        # completed_workouts
        self.upsert_workout_bpm(
            once,
            processing_time
        )

        self._await_queries(once)

        print(
            f"Silver completed in "
            f"{int(time.time()) - start} seconds"
        )
    #########################################################################################
    def assert_count(
        self,
        table_name,
        expected_count,
        filter_condition="true"
    ):

        print(
            f"Validating record counts in {table_name}...",
            end=""
        )

        actual_count = (
            spark.read
            .table(
                f"{self.catalog}."
                f"{self.silver}."
                f"{table_name}"
            )
            .where(filter_condition)
            .count()
        )

        assert actual_count == expected_count, (
            f"Expected {expected_count:,} records, "
            f"found {actual_count:,} "
            f"in {table_name}"
        )

        print(
            f"Found {actual_count:,} / "
            f"Expected {expected_count:,}: Success"
        )

    ###################################################################
    def validate(self, sets=1):

        import time

        start = int(time.time())

        print(
            "\nValidating Silver layer records..."
        )

        self.assert_count(
            "users",
            5 if sets == 1 else 10
        )

        self.assert_count(
            "gym_logs",
            8 if sets == 1 else 16
        )

        self.assert_count(
            "user_profile",
            5 if sets == 1 else 10
        )

        self.assert_count(
            "workouts",
            16 if sets == 1 else 32
        )

        self.assert_count(
            "heart_rate",
            sets * 253801
        )

        self.assert_count(
            "user_bins",
            5 if sets == 1 else 10
        )

        self.assert_count(
            "completed_workouts",
            8 if sets == 1 else 16
        )

        self.assert_count(
            "workout_bpm",
            3968 if sets == 1 else 8192
        )

        print(
            f"Silver layer validation completed in "
            f"{int(time.time()) - start} seconds"
        )




### Description de la classe `Silver`

La classe `Silver` contient toute la logique de traitement de la **couche Silver** du projet.

Son rôle est de prendre les données brutes présentes dans la couche Bronze, puis de les rendre plus propres, plus fiables et plus utiles pour la suite du pipeline.

Le principe général est :

`Bronze → lecture incrémentale → déduplication → transformation → MERGE → Silver`

Concrètement, la classe lit uniquement les nouvelles données grâce à Spark Structured Streaming et aux checkpoints. Elle évite donc de retraiter continuellement les mêmes enregistrements.

Elle contient une fonction pour chaque table Silver importante du projet :

- `upsert_users()` : construit la table des utilisateurs à partir des inscriptions présentes en Bronze.
- `upsert_gym_logs()` : construit les entrées et sorties des utilisateurs dans les salles de sport.
- `upsert_user_profile()` : traite les événements CDC du profil utilisateur et conserve les informations les plus récentes.
- `upsert_workouts()` : transforme les événements Kafka de début et de fin d’entraînement.
- `upsert_heart_rate()` : extrait les mesures BPM et identifie les valeurs valides ou invalides.
- `upsert_user_bins()` : enrichit les utilisateurs avec une tranche d’âge, le genre et la localisation.
- `upsert_completed_workouts()` : rapproche les événements `start` et `stop` afin de reconstruire une séance complète.
- `upsert_workout_bpm()` : associe les mesures cardiaques aux séances pendant lesquelles elles ont été enregistrées.

Par exemple :

`Kafka BPM → Bronze → heart_rate → workout_bpm`

ou :

`Workout start + Workout stop → completed_workouts`

La méthode `upsert()` sert de point d’entrée principal : elle lance les différents traitements Silver dans le bon ordre, car certaines tables dépendent d’autres tables déjà créées.

Enfin, `validate()` permet de vérifier que les tables Silver contiennent bien le nombre de lignes attendu après le traitement.

L’objectif de cette classe est donc de transformer les données brutes de la couche Bronze en **données nettoyées, dédupliquées, enrichies et cohérentes**, prêtes à être utilisées pour construire la couche Gold.

### Registered users → `users`

Cette fonction lit les nouvelles inscriptions depuis la table Bronze `registered_users_bz`, supprime les doublons puis alimente la table Silver `users`.

Le flux est :

`registered_users_bz → déduplication → transformation → MERGE → users`

Elle fonctionne en streaming avec checkpoint afin de ne traiter que les nouvelles données.

### Gym Bronze → `gym_logs`

Cette fonction lit les nouvelles données de connexion depuis `gym_logins_bz`, supprime les doublons puis alimente la table Silver `gym_logs`.

Le `MERGE` permet aussi de mettre à jour l’heure de sortie `logout` lorsqu’une information plus récente arrive.

Le flux est :

`gym_logins_bz → déduplication → MERGE → gym_logs`

### CDC User Profile → `user_profile`

Cette fonction lit les événements `user_info` depuis `kafka_multiplex_bz`, extrait les données JSON et garde la version la plus récente de chaque utilisateur.

Elle applique ensuite un `MERGE` pour insérer les nouveaux profils ou mettre à jour les profils existants.

Le flux est :

`kafka_multiplex_bz → CDC → dernière version → MERGE → user_profile`

### Workout events → `workouts`

Cette fonction lit les événements `workout` depuis `kafka_multiplex_bz`, extrait les données JSON puis supprime les doublons.

Elle enregistre ensuite les événements de début et de fin d’entraînement dans la table Silver `workouts`.

Le flux est :

`kafka_multiplex_bz → workout → déduplication → MERGE → workouts`

### BPM → `heart_rate`

Cette fonction récupère les événements `bpm` depuis `kafka_multiplex_bz`, extrait les mesures de fréquence cardiaque et supprime les doublons.

Elle marque aussi les valeurs de BPM invalides (`<= 0`) avant de les enregistrer dans la table Silver `heart_rate`.

Le flux est :

`kafka_multiplex_bz → bpm → validation → déduplication → MERGE → heart_rate`

### Tranches d'âge → `user_bins`

Cette partie calcule l’âge de chaque utilisateur à partir de sa date de naissance `dob`, puis le classe dans une tranche d’âge.

Exemples :

`22 ans → 18-25`  
`31 ans → 25-35`  
`67 ans → 65-75`

Ces informations sont ensuite utilisées pour alimenter la table Silver `user_bins`.

Le flux est :

`user_profile → calcul de l'âge → tranche d'âge → user_bins`

### User profile → `user_bins`

Cette fonction utilise les données de `user_profile` pour enrichir les utilisateurs avec leur tranche d’âge, leur genre, leur ville et leur état.

Elle met ensuite à jour la table Silver `user_bins` avec un `MERGE`.

Le flux est :

`user_profile + users → enrichissement → MERGE → user_bins`

### Associer START et STOP → `completed_workouts`

Cette fonction associe les événements de début `start` et de fin `stop` d’un même entraînement.

Elle permet de reconstruire une séance complète avec :

`start_time → end_time`

Puis le résultat est enregistré dans la table Silver `completed_workouts`.

Le flux est :

`workouts → start + stop → séance complète → completed_workouts`

### Entraînement + BPM → `workout_bpm`

Cette fonction associe les mesures de fréquence cardiaque `heart_rate` aux séances terminées dans `completed_workouts`.

Elle garde uniquement les BPM enregistrés entre le début et la fin de chaque entraînement.

Le flux est :

`completed_workouts + heart_rate → association par période → workout_bpm`

### Lancer toute la couche Silver

La méthode `upsert()` sert à exécuter l’ensemble des traitements Silver dans le bon ordre.

Elle lance d’abord les tables principales, puis les tables qui dépendent des résultats précédents.

Le flux est :

`Bronze → traitements Silver → tables intermédiaires → workout_bpm`

La méthode `_await_queries()` attend la fin des streams avant de passer à l’étape suivante afin de respecter les dépendances entre les tables.